# Lab 00 — UTD19 Full-Release + MFD Reconstruction (Colab)

**Run order is intentional.** Full released dataset audit comes before city selection. Raw sanity MFDs are not labelled as exact paper reproduction. Heavy data stays in Google Drive.

In [ ]:
!pip -q install "duckdb>=1.1,<2" "pandas>=2.2,<3" "numpy>=1.26,<3" "scipy>=1.12,<2" "matplotlib>=3.8,<4" "pyyaml>=6,<7" "pyarrow>=15,<25" "pytest>=8,<9"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib

# Change only if Claude/user placed the repo elsewhere.
CANDIDATES = [
    Path('/content/DhakaFlow-Nexus-Labs-00-01'),
    Path('/content/drive/MyDrive/DhakaFlow-Nexus-Labs-00-01'),
]
REPO_ROOT = next((p for p in CANDIDATES if p.exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Upload/unzip DhakaFlow-Nexus-Labs-00-01 to /content or MyDrive first.')
LAB = REPO_ROOT/'paper_labs/00_utd19'
DRIVE = Path('/content/drive/MyDrive/DhakaFlow-Nexus-Data/Lab00_UTD19')
RAW = DRIVE/'raw'; OUT = DRIVE/'outputs'
for p in [RAW, OUT/'audit', OUT/'cities', OUT/'models', OUT/'reports', OUT/'figures']:
    p.mkdir(parents=True, exist_ok=True)
print('REPO_ROOT=', REPO_ROOT)
print('DRIVE=', DRIVE)
print('LOCAL_UTD19_LARGE_DATA_PRESENT=', any((REPO_ROOT).glob('**/utd19_u.csv')))

## 1. Fetch open paper resources
This downloads only papers/supplementary files, not the 7GB-class UTD19 measurement data.

In [ ]:
subprocess.run([sys.executable, str(LAB/'paper/fetch_open_resources.py')], check=False)

## 2. Locate official UTD19 files in Drive
Get access from https://utd19.ethz.ch/ and place the official files in `MyDrive/DhakaFlow-Nexus-Data/Lab00_UTD19/raw/`. If the official emailed download URL is stored in Colab Secrets as `UTD19_DOWNLOAD_URL`, Claude may add a downloader after verifying the archive/file format. Do not bypass access terms.

In [ ]:
MEASURE = RAW/'utd19_u.csv'
DETECTORS = next((RAW/n for n in ['detectors_public.csv','detectors.csv'] if (RAW/n).exists()), None)
LINKS = RAW/'links.csv'
print('measurement:', MEASURE, MEASURE.exists(), MEASURE.stat().st_size if MEASURE.exists() else None)
print('detectors:', DETECTORS)
print('links:', LINKS, LINKS.exists())
if not MEASURE.exists():
    raise FileNotFoundError('Official utd19_u.csv missing in Drive. Put it in RAW path before continuing.')

## 3. Verify source code + synthetic tests before full scan

In [ ]:
env=os.environ.copy(); env['PYTHONPATH']=str(REPO_ROOT)
subprocess.run([sys.executable,'-m','pytest','-q',str(LAB/'tests')],check=True,env=env)

## 4. FULL RELEASE AUDIT — all cities, no random city selection

In [ ]:
sys.path.insert(0,str(LAB))
from src.audit import full_city_audit, deterministic_candidate_score
audit, error_values = full_city_audit(MEASURE, OUT/'audit/city_audit.csv')
ranked = deterministic_candidate_score(audit)
ranked.to_csv(OUT/'audit/city_candidates.csv', index=False)
error_values.to_csv(OUT/'audit/error_values.csv', index=False)
display(ranked)
display(error_values.head(100))

## 5. Confirm error-flag semantics before paper-faithful cleaning
Claude must read the UTD19 manual / `.txt` field descriptions and write the meaning here. **Do not proceed to claim paper reproduction until confirmed.** Raw reconstruction below intentionally does not silently interpret error flags.

In [ ]:
print('Distinct error flag values by city are saved at', OUT/'audit/error_values.csv')
print('ACTION FOR CLAUDE: document error semantics in DATA_SPEC.md before final PASS.')

## 6. Deterministic smoke reconstruction
The city is selected by the transparent coverage score produced above, not by convenience/random choice. This is a validation run before the all-city run.

In [ ]:
CITY = str(ranked.iloc[0].city)
print('Deterministically selected smoke city:', CITY)
cmd=[sys.executable,'-m','src.run_full',
     '--config',str(LAB/'configs/lab00.yaml'),
     '--measurements',str(MEASURE),
     '--out',str(OUT/'smoke'),
     '--city',CITY,'--max-days','7']
subprocess.run(cmd,check=True,cwd=LAB,env={**os.environ,'PYTHONPATH':str(LAB)})

## 7. Full selected-city run — all available days

In [ ]:
cmd=[sys.executable,'-m','src.run_full',
     '--config',str(LAB/'configs/lab00.yaml'),
     '--measurements',str(MEASURE),
     '--out',str(OUT/'selected_city_full'),
     '--city',CITY]
subprocess.run(cmd,check=True,cwd=LAB,env={**os.environ,'PYTHONPATH':str(LAB)})

## 8. Full released-dataset RAW aggregation — every city, out-of-core
This scans the **entire released measurement file** with DuckDB and writes city/day/interval aggregates to Parquet without loading each full city into RAM. This product is explicitly `RAW_SANITY_MFD`, not the paper's 107-region reconstruction.

In [ ]:
FULL_RAW = OUT/'full_release/RAW_SANITY_MFD_all_cities.parquet'
FULL_RAW.parent.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,'-m','src.full_release_raw','--measurements',str(MEASURE),'--out',str(FULL_RAW),
     '--min-detectors','10']
subprocess.run(cmd,check=True,cwd=LAB,env={**os.environ,'PYTHONPATH':str(LAB)})

## 9. Paper's 107 regional networks
Exact paper reproduction requires the 2019 regional partitions, network length scaling, and historical calibration. Claude must extract/verify region information from the supplementary material and populate `configs/regions.yaml`. If that cannot be recovered, status remains **PARTIAL paper-faithful reconstruction**, even though full public UTD19 was processed.

In [ ]:
regions = LAB/'configs/regions.yaml'
print('regions config exists?', regions.exists())
print('Do NOT invent missing region definitions.')

## 10. Manifest / final report
Claude should copy only lightweight summaries/plots back to the repo, never raw UTD19. The Drive output tree is the authoritative heavy-output location.

In [ ]:
def sha256(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
manifest={'measurement':{'path':str(MEASURE),'bytes':MEASURE.stat().st_size,'sha256':sha256(MEASURE)},
          'repo':str(REPO_ROOT),'drive':str(DRIVE),'selected_city':CITY}
(OUT/'reports/final_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))